## Getting Population Data

In [4]:
import pandas as pd

In [1]:
import requests

url = "https://api.census.gov/data/2023/acs/acs1"

API_KEY = "3d1a8efb010ab94526aed9bb9b1b8ba58c722e3d"

states = ["09", "34", "36"]  # CT, NJ, NY

all_data = []

for s in states:
    params = {
        "get": "NAME,B01003_001E",
        "for": "county:*",
        "in": f"state:{s}",
        "key": API_KEY
    }

    r = requests.get(url, params=params)

    print("\nSTATE:", s)
    print("STATUS:", r.status_code)
    print("CONTENT (first 300 chars):")
    print(r.text[:300])
    print("-" * 50)


STATE: 09
STATUS: 200
CONTENT (first 300 chars):
[["NAME","B01003_001E","state","county"],
["Capitol Planning Region, Connecticut","975328","09","110"],
["Greater Bridgeport Planning Region, Connecticut","327651","09","120"],
["Lower Connecticut River Valley Planning Region, Connecticut","176215","09","130"],
["Naugatuck Valley Planning Region, Co
--------------------------------------------------

STATE: 34
STATUS: 200
CONTENT (first 300 chars):
[["NAME","B01003_001E","state","county"],
["Atlantic County, New Jersey","275213","34","001"],
["Bergen County, New Jersey","957736","34","003"],
["Burlington County, New Jersey","469167","34","005"],
["Camden County, New Jersey","527196","34","007"],
["Cape May County, New Jersey","94610","34","009
--------------------------------------------------

STATE: 36
STATUS: 200
CONTENT (first 300 chars):
[["NAME","B01003_001E","state","county"],
["Albany County, New York","316659","36","001"],
["Bronx County, New York","1356476","36","005"],
["Broo

In [5]:
pop = pd.read_csv("../data/co-est2025-alldata.csv", encoding = "latin1")

In [6]:
print(pop.columns.tolist())

['SUMLEV', 'REGION', 'DIVISION', 'STATE', 'COUNTY', 'STNAME', 'CTYNAME', 'ESTIMATESBASE2020', 'POPESTIMATE2020', 'POPESTIMATE2021', 'POPESTIMATE2022', 'POPESTIMATE2023', 'POPESTIMATE2024', 'POPESTIMATE2025', 'NPOPCHG2020', 'NPOPCHG2021', 'NPOPCHG2022', 'NPOPCHG2023', 'NPOPCHG2024', 'NPOPCHG2025', 'BIRTHS2020', 'BIRTHS2021', 'BIRTHS2022', 'BIRTHS2023', 'BIRTHS2024', 'BIRTHS2025', 'DEATHS2020', 'DEATHS2021', 'DEATHS2022', 'DEATHS2023', 'DEATHS2024', 'DEATHS2025', 'NATURALCHG2020', 'NATURALCHG2021', 'NATURALCHG2022', 'NATURALCHG2023', 'NATURALCHG2024', 'NATURALCHG2025', 'INTERNATIONALMIG2020', 'INTERNATIONALMIG2021', 'INTERNATIONALMIG2022', 'INTERNATIONALMIG2023', 'INTERNATIONALMIG2024', 'INTERNATIONALMIG2025', 'DOMESTICMIG2020', 'DOMESTICMIG2021', 'DOMESTICMIG2022', 'DOMESTICMIG2023', 'DOMESTICMIG2024', 'DOMESTICMIG2025', 'NETMIG2020', 'NETMIG2021', 'NETMIG2022', 'NETMIG2023', 'NETMIG2024', 'NETMIG2025', 'RESIDUAL2020', 'RESIDUAL2021', 'RESIDUAL2022', 'RESIDUAL2023', 'RESIDUAL2024', 'RES

In [7]:
# Filtering for NY-NJ-CT
pop = pop[
    pop["STATE"].isin([9, 34, 36])
]

In [8]:
pop = pop[
    pop["COUNTY"] > 0
]

In [9]:
pop["fips"] = (
    pop["STATE"].astype(str).str.zfill(2)
    + pop["COUNTY"].astype(str).str.zfill(3)
)

In [10]:
pop = pop[[
    "fips",
    "STNAME",
    "CTYNAME",
    "POPESTIMATE2023"
]]

In [11]:
# Renaming columns
pop = pop.rename(columns={
    "STNAME": "state",
    "CTYNAME": "county",
    "POPESTIMATE2023": "population"
})

In [12]:
# Check data
print(pop.shape)
print(pop.head())

(92, 4)
      fips        state                                          county  \
316  09110  Connecticut                         Capitol Planning Region   
317  09120  Connecticut              Greater Bridgeport Planning Region   
318  09130  Connecticut  Lower Connecticut River Valley Planning Region   
319  09140  Connecticut                Naugatuck Valley Planning Region   
320  09150  Connecticut        Northeastern Connecticut Planning Region   

     population  
316      981775  
317      332081  
318      176419  
319      457384  
320       96790  


In [13]:
pop = pop[pop['state'] != 'Connecticut']

In [15]:
ct_towns = pd.read_csv("../reference/ct_crosswalk/ct_town_crosswalk.csv")

In [16]:
# Load the population data
pop_df = pd.read_csv('../data/ct_towns_pop2023.csv')

# Standardize town names for merging (uppercase to match ct_towns)
pop_df['town_name'] = pop_df['town_name'].str.upper()

# Merge with ct_towns to get county info
merged = ct_towns[['town_name', 'county_name']].merge(
    pop_df,
    on='town_name',
    how='left'
)

# Group by county and sum populations
county_pop = merged.groupby('county_name')['pop_2023'].sum().reset_index()
county_pop.columns = ['county_name', 'total_pop_2023']
county_pop = county_pop.sort_values('total_pop_2023', ascending=False)

print(county_pop)

         county_name  total_pop_2023
0   Fairfield County             0.0
1    Hartford County             0.0
2  Litchfield County             0.0
3   Middlesex County             0.0
4   New Haven County             0.0
5  New London County             0.0
6     Tolland County             0.0
7     Windham County             0.0


In [17]:
# Normalize both to uppercase for matching
ct_towns_copy = ct_towns.copy()
ct_towns_copy['town_name_upper'] = ct_towns_copy['town_name'].str.upper()
pop_df['town_name_upper'] = pop_df['town_name'].str.upper()

# Merge on the normalized column
merged = ct_towns_copy[['town_name', 'town_name_upper', 'county_name']].merge(
    pop_df[['town_name_upper', 'pop_2023']],
    on='town_name_upper',
    how='left'
)

# Check unmatched
unmatched = merged[merged['pop_2023'].isna()]['town_name'].tolist()
print("Unmatched towns:", unmatched)

# Group by county
county_pop = merged.groupby('county_name')['pop_2023'].sum().reset_index()
county_pop.columns = ['county_name', 'total_pop_2023']
county_pop = county_pop.sort_values('total_pop_2023', ascending=False)
print(county_pop)

Unmatched towns: []
         county_name  total_pop_2023
0   Fairfield County          963780
1    Hartford County          898478
4   New Haven County          865717
5  New London County          268518
2  Litchfield County          186551
3   Middlesex County          166110
6     Tolland County          150906
7     Windham County          117116


In [18]:
# Get unique county_fips + county_name from ct_towns
county_fips_map = ct_towns[['county_fips', 'county_name']].drop_duplicates()

# Merge fips into county population df
ct_county_pop = county_pop.merge(
    county_fips_map,
    on='county_name',
    how='left'
)
ct_county_pop.set_index('county_fips', inplace = True)
print(ct_county_pop)

                   county_name  total_pop_2023
county_fips                                   
9001          Fairfield County          963780
9003           Hartford County          898478
9009          New Haven County          865717
9011         New London County          268518
9005         Litchfield County          186551
9007          Middlesex County          166110
9013            Tolland County          150906
9015            Windham County          117116


In [19]:
ct_county_pop['state'] = 'Connecticut'
pop['fips'] = pop['fips'].astype(int)
pop.set_index('fips', inplace = True)

In [22]:
ct_county_pop = ct_county_pop.rename(columns={'county_name': 'county', 'total_pop_2023':'population'})
ct_county_pop.index.name = 'fips'
ct_county_pop.head()

,county,population,state
fips,,,
9001,Fairfield County,963780,Connecticut
9003,Hartford County,898478,Connecticut
9009,New Haven County,865717,Connecticut
9011,New London County,268518,Connecticut
9005,Litchfield County,186551,Connecticut


In [23]:
pop_all = pd.concat([ct_county_pop, pop])

In [24]:
pop_all.head(20)

,county,population,state
fips,,,
9001,Fairfield County,963780,Connecticut
9003,Hartford County,898478,Connecticut
9009,New Haven County,865717,Connecticut
9011,New London County,268518,Connecticut
9005,Litchfield County,186551,Connecticut
9007,Middlesex County,166110,Connecticut
9013,Tolland County,150906,Connecticut
9015,Windham County,117116,Connecticut
34001,Atlantic County,276643,New Jersey


## Importing Stroke centers with fips

In [28]:
nj_stroke_centers = pd.read_csv("../data/geographic_accessibility_data/nj_all_stroke_centers_geocoded_with_fips.csv")
ny_stroke_centers = pd.read_csv("../data/geographic_accessibility_data/ny_all_stroke_centers_geocoded_with_fips.csv")

### FIlling in missing values for NJ

In [38]:
nj_stroke_centers.head(20)

,name,designation,address,latitude,longitude,fips
0,AtlanticCare Regional Medical Center,Comprehensive,"1925 Pacific Ave, Atlantic City, NJ 08401",39.357959,-74.433683,34001.0
1,Valley Hospital,Comprehensive,"223 N Van Dien Ave, Ridgewood, NJ 07450",40.982828,-74.101808,34003.0
2,Hackensack University Medical Center,Comprehensive,"30 Prospect Ave, Hackensack, NJ 07601",40.884467,-74.057638,34003.0
3,Cooper University Hospital,Comprehensive,"1 Cooper Plaza, Camden, NJ 08103",39.940854,-75.115774,34007.0
4,Our Lady of Lourdes Medical Center,Comprehensive,"1600 Haddon Ave, Camden, NJ 08103",NaN,NaN,34007.0
5,University Hospital,Comprehensive,"150 Bergen St, Newark, NJ 07103",40.739774,-74.192687,34013.0
6,Saint Barnabas Medical Center,Comprehensive,"94 Old Short Hills Rd, Livingston, NJ 07039",40.765071,-74.301965,34013.0
7,Jefferson Washington Township Hospital,Comprehensive,"435 Hurffville-Cross Keys Rd, Turnersville, NJ...",39.733641,-75.064734,34015.0
8,Capital Health System at Fuld,Comprehensive,"750 Brunswick Ave, Trenton, NJ 08638",40.236022,-74.752690,34021.0
9,Robert Wood Johnson University Hospital New Br...,Comprehensive,"1 Robert Wood Johnson Pl, New Brunswick, NJ 08901",NaN,NaN,NaN


In [41]:
nan_rows = nj_stroke_centers[nj_stroke_centers['fips'].isna()]
print(nan_rows['name'].tolist())

['Robert Wood Johnson University Hospital New Brunswick', 'Morristown Memorial Hospital', 'AtlanticCare Regional Medical Center - Jimmie Leeds Road', 'Englewood Hospital', 'Lourdes Medical Center Burlington County', 'East Orange General Hospital', 'Hackensack UMC - Mountainside', 'Capital Health Medical Center - Hopewell', 'Robert Wood Johnson University Hospital at Hamilton', 'Raritan Bay Medical Center Old Bridge']


In [43]:
# Placeholder dict — fill in with the correct FIPS codes for each facility
fips_lookup = {
    "Robert Wood Johnson University Hospital New Brunswick": 34019,
    "Morristown Memorial Hospital": 34027,
    "AtlanticCare Regional Medical Center - Jimmie Leeds Road": 34001,
    "Englewood Hospital": 34003,
    'Lourdes Medical Center Burlington County': 34007,
    'East Orange General Hospital' : 34013,
    "Hackensack UMC - Mountainside": 34013,  # Montclair → Essex County
    "Capital Health Medical Center - Hopewell": 34021,  # Pennington → Mercer County
    "Robert Wood Johnson University Hospital at Hamilton": 34021,  # Hamilton → Mercer County
    "Raritan Bay Medical Center Old Bridge": 34023,  # Old Bridge → Middlesex County 
}

# Get the rows with missing fips, so you can confirm names match exactly
nan_rows = nj_stroke_centers[nj_stroke_centers['fips'].isna()]
print(nan_rows['name'].tolist())

# Sanity check: make sure every NaN row's name is covered in the dict
missing_from_lookup = set(nan_rows['name']) - set(fips_lookup.keys())
assert not missing_from_lookup, f"These names aren't in fips_lookup: {missing_from_lookup}"

# Fill in using the name as the join key
nj_stroke_centers['fips'] = nj_stroke_centers['fips'].fillna(nj_stroke_centers['name'].map(fips_lookup))

['Robert Wood Johnson University Hospital New Brunswick', 'Morristown Memorial Hospital', 'AtlanticCare Regional Medical Center - Jimmie Leeds Road', 'Englewood Hospital', 'Lourdes Medical Center Burlington County', 'East Orange General Hospital', 'Hackensack UMC - Mountainside', 'Capital Health Medical Center - Hopewell', 'Robert Wood Johnson University Hospital at Hamilton', 'Raritan Bay Medical Center Old Bridge']


In [46]:
# Check for missing fips values
missing_fips = nj_stroke_centers[nj_stroke_centers['fips'].isna()]

print(f"Number of rows with missing fips: {len(missing_fips)}")
print(missing_fips[['name', 'address']])

Number of rows with missing fips: 0
Empty DataFrame
Columns: [name, address]
Index: []


## Fill in missing values for NY

In [47]:
fips_lookup = {
    "Garnet Health Medical Center - Catskills Harris Campus": 36105,  # Harris → Sullivan County
    "Guthrie Corning Hospital": 36101,  # Corning → Steuben County
    "John T Mather Memorial Hospital": 36103,  # Port Jefferson → Suffolk County
    "Mercy Hospital": 36059,  # Rockville Centre → Nassau County
    "Mount Sinai South Nassau Hospital": 36059,  # Oceanside → Nassau County
}
nan_rows = ny_stroke_centers[ny_stroke_centers['fips'].isna()]
print(nan_rows['name'].tolist())

# Sanity check: make sure every NaN row's name is covered in the dict
missing_from_lookup = set(nan_rows['name']) - set(fips_lookup.keys())
assert not missing_from_lookup, f"These names aren't in fips_lookup: {missing_from_lookup}"

# Fill in using the name as the join key
ny_stroke_centers['fips'] = ny_stroke_centers['fips'].fillna(ny_stroke_centers['name'].map(fips_lookup))

['Garnet Health Medical Center - Catskills Harris Campus', 'Guthrie Corning Hospital', 'John T Mather Memorial Hospital', 'Mercy Hospital', 'Mount Sinai South Nassau Hospital']


In [48]:
# Check for missing fips values
missing_fips = nj_stroke_centers[nj_stroke_centers['fips'].isna()]

print(f"Number of rows with missing fips: {len(missing_fips)}")
print(missing_fips[['name', 'address']])

Number of rows with missing fips: 0
Empty DataFrame
Columns: [name, address]
Index: []


## Importing CT

In [57]:
ct_advanced = pd.read_csv("../data/geographic_accessibility_data/ct_advanced_geocoded_with_fips.csv")
ct_basic = pd.read_csv("../data/geographic_accessibility_data/ct_basic_geocoded_with_fips.csv")

In [58]:
ct_basic.head()

,name,group,latitude,longitude,fips
0,Bridgeport Hospital Milford Campus,Basic,41.216545,-73.065360,9009
1,Charlotte Hungerford Hospital,Basic,41.792271,-73.133769,9005
2,Day Kimball Hospital,Basic,41.906093,-71.913028,9015
3,Greenwich Hospital,Basic,41.034299,-73.630646,9001
4,Griffin Hospital,Basic,41.336435,-73.090512,9009


In [59]:
ct_advanced.head()

,name,group,latitude,longitude,fips
0,Bridgeport Hospital,Advanced,41.189471,-73.167470,9001
1,Danbury Hospital,Advanced,41.405218,-73.445159,9001
2,Hartford Hospital,Advanced,41.754069,-72.679390,9003
3,Norwalk Hospital,Advanced,41.110829,-73.421953,9001
4,Saint Francis Hospital and Medical Center,Advanced,41.774141,-72.698084,9003


In [86]:
ct_stroke_centers = pd.concat([ct_advanced, ct_basic])


In [87]:
ct_stroke_centers.head()

,name,group,latitude,longitude,fips
0,Bridgeport Hospital,Advanced,41.189471,-73.167470,9001
1,Danbury Hospital,Advanced,41.405218,-73.445159,9001
2,Hartford Hospital,Advanced,41.754069,-72.679390,9003
3,Norwalk Hospital,Advanced,41.110829,-73.421953,9001
4,Saint Francis Hospital and Medical Center,Advanced,41.774141,-72.698084,9003


In [88]:
ct_stroke_centers['state'] = 'Connecticut'
ct_stroke_centers.drop(columns=['latitude', 'longitude'], inplace = True) 
ct_stroke_centers.head()

,name,group,fips,state
0,Bridgeport Hospital,Advanced,9001,Connecticut
1,Danbury Hospital,Advanced,9001,Connecticut
2,Hartford Hospital,Advanced,9003,Connecticut
3,Norwalk Hospital,Advanced,9001,Connecticut
4,Saint Francis Hospital and Medical Center,Advanced,9003,Connecticut


In [89]:
ct_stroke_centers.rename(columns={"group": "designation"}, inplace=True)

In [80]:
ct_stroke_centers['fips'] = ct_stroke_centers['fips'].astype(str).str.zfill(5)
ct_stroke_centers

,name,designation,fips,state
0,Bridgeport Hospital,Advanced,09001,Connecticut
1,Danbury Hospital,Advanced,09001,Connecticut
2,Hartford Hospital,Advanced,09003,Connecticut
3,Norwalk Hospital,Advanced,09001,Connecticut
4,Saint Francis Hospital and Medical Center,Advanced,09003,Connecticut
5,St. Vincent's Medical Center,Advanced,09001,Connecticut
6,Yale New Haven Hospital,Advanced,09009,Connecticut
0,Bridgeport Hospital Milford Campus,Basic,09009,Connecticut
1,Charlotte Hungerford Hospital,Basic,09005,Connecticut
2,Day Kimball Hospital,Basic,09015,Connecticut


### Cleaning and Merging

In [70]:
nj_stroke_centers.drop(columns=['latitude', 'longitude'], inplace = True)
ny_stroke_centers.drop(columns=['latitude', 'longitude'], inplace = True)

In [72]:
nj_stroke_centers.drop(columns=['address'], inplace = True)
ny_stroke_centers.drop(columns=['address'], inplace = True)

In [73]:
nj_stroke_centers.head()

,name,designation,fips
0,AtlanticCare Regional Medical Center,Comprehensive,34001.0
1,Valley Hospital,Comprehensive,34003.0
2,Hackensack University Medical Center,Comprehensive,34003.0
3,Cooper University Hospital,Comprehensive,34007.0
4,Our Lady of Lourdes Medical Center,Comprehensive,34007.0


In [78]:
nj_stroke_centers['state'] = 'New Jersey'
ny_stroke_centers['state'] = 'New York'

In [90]:
stroke_centers = pd.concat([ct_stroke_centers, nj_stroke_centers, ny_stroke_centers])
stroke_centers

,name,designation,fips,state
0,Bridgeport Hospital,Advanced,9001.0,Connecticut
1,Danbury Hospital,Advanced,9001.0,Connecticut
2,Hartford Hospital,Advanced,9003.0,Connecticut
3,Norwalk Hospital,Advanced,9001.0,Connecticut
4,Saint Francis Hospital and Medical Center,Advanced,9003.0,Connecticut
...,...,...,...,...
117,Syosset Hospital,Primary Stroke Center,36059.0,New York
118,Unity Hospital of Rochester,Primary Stroke Center,36055.0,New York
119,United Memorial Medical Center North Street Ca...,Primary Stroke Center,36037.0,New York
120,University Hospital of Brooklyn (SUNY Downstate),Primary Stroke Center,36047.0,New York


In [82]:
#converting types
stroke_centers['fips'] = stroke_centers['fips'].astype(float).astype(int).astype(str).str.zfill(5)
stroke_centers

,name,designation,fips,state
0,Bridgeport Hospital,Advanced,09001,Connecticut
1,Danbury Hospital,Advanced,09001,Connecticut
2,Hartford Hospital,Advanced,09003,Connecticut
3,Norwalk Hospital,Advanced,09001,Connecticut
4,Saint Francis Hospital and Medical Center,Advanced,09003,Connecticut
...,...,...,...,...
117,Syosset Hospital,Primary Stroke Center,36059,New York
118,Unity Hospital of Rochester,Primary Stroke Center,36055,New York
119,United Memorial Medical Center North Street Ca...,Primary Stroke Center,36037,New York
120,University Hospital of Brooklyn (SUNY Downstate),Primary Stroke Center,36047,New York
